# Disaster Tweets — exploratory analysis

Loads the raw Kaggle data, runs the cleaning pipeline, and looks at the
signal that is actually available before any model is trained.

Run `pip install -r requirements.txt` first.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # import the repo package from notebooks/

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from preprocessor import Preprocessor

sns.set_theme(style='whitegrid')
ROOT = Path.cwd().parent


## Load the raw data


In [ ]:
train = pd.read_csv(ROOT / 'dataset' / 'train.csv')
test = pd.read_csv(ROOT / 'dataset' / 'test.csv')

print(f'train {train.shape}   test {test.shape}')
print(f"positive rate: {train['target'].mean():.1%}")
train.head()


In [ ]:
# How much of each column is actually populated?
train.isna().mean().to_frame('missing share').style.format('{:.1%}')


## Clean the text

`Preprocessor` bundles the cleaning steps behind boolean flags. The
aggressive settings below suit a bag-of-words model; a pretrained
transformer wants far lighter cleaning (see `prepare_data.py`).


In [ ]:
pre = Preprocessor()

options = dict(
    lowercase=True, contractions=True, urls=True, punctuation=True,
    html_tags=True, emoji=True, spelling=True, abbreviations=True, lemma=True,
)

sample = train.head(5)['text']
for raw in sample:
    print('RAW :', raw)
    print('CLEAN:', pre.process_text(raw, **options))
    print()


In [ ]:
clean = train.copy()
clean['text'] = pre.process_series(train['text'], **options)
clean['location'] = pre.process_series(train['location'], **options)
clean.head()


## Tweet length


In [ ]:
train['length'] = train['text'].str.len()

fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(data=train, x='length', hue='target', bins=40, ax=ax)
ax.set_title('Character length by class')
plt.show()

train.groupby('target')['length'].describe()[['mean', '50%', 'max']]


## Keyword is the strongest cheap feature

Only ~0.8% of rows are missing `keyword`, and the disaster rate per keyword
spans nearly the whole 0–100% range. That is why both training scripts
prepend it to the tweet text.


In [ ]:
keyword_stats = (train.dropna(subset=['keyword'])
                 .groupby('keyword')['target']
                 .agg(['mean', 'size'])
                 .query('size >= 20')
                 .sort_values('mean'))

extremes = pd.concat([keyword_stats.head(12), keyword_stats.tail(12)])

fig, ax = plt.subplots(figsize=(8, 8))
extremes['mean'].plot(kind='barh', ax=ax)
ax.set_title('Disaster rate by keyword (12 lowest and 12 highest)')
ax.set_xlabel('share labelled as a real disaster')
plt.tight_layout()
plt.show()


## Location is mostly noise

A third of the rows have no location, and the free-text values barely repeat,
so there is little to generalise from.


In [ ]:
counts = train['location'].value_counts()
print(f'{train["location"].isna().mean():.1%} missing, '
      f'{counts.size} distinct values, '
      f'{(counts == 1).mean():.1%} of them appearing exactly once')

fig, ax = plt.subplots(figsize=(10, 4))
counts.head(20).plot(kind='bar', ax=ax)
ax.set_title('20 most common locations')
plt.tight_layout()
plt.show()


## Next steps

```bash
python prepare_data.py --preset bow    # cache the cleaned CSVs
python train_baseline.py               # 5-fold CV + submission.csv
```
